In [ ]:
!pip install kagglehub
import kagglehub

path = kagglehub.dataset_download("mohankrishnathalla/student-academic-performance-and-dropout-risk")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd

files=os.listdir(path)
print('file yang tersedia: ', files)

file_csv= os.path.join(path, files[0])
df=pd.read_csv(file_csv)
pd.set_option('display.max_column',None)

df.isnull().sum()
df.drop_duplicates(inplace=True)
df.dropna(inplace=True)
print(df.columns.tolist())
display(df.describe())
display(df.head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score

x = df[['assignments_submitted_rate', 'units_failed', 'attendance_rate']]
y= df['dropout']


x_train,x_test,y_train,y_test= train_test_split(x,y, test_size=0.25, random_state=42)

model= RandomForestClassifier(random_state=42)
model.fit(x_train,y_train)

y_pred=model.predict(x_test)
print('akurasi skor: ', accuracy_score(y_test, y_pred))
print('clasiification report:\n',classification_report(y_test,y_pred))
print('confusion matriks: \n', confusion_matrix(y_test,y_pred))


kls 0 (aman), precision 0.79 berati emg yang beneran aman 79%, trs recall 0.83 itu berati yang aman, dan sisanya ditebak dropout (17%), trs f1 score itu rata" dari precision dan recall

kls 1 (dropout), precision 0.59 berati 59% pasti dropout, trs recall 0.53 berati 53% itu yang dropout, sisanya ditebak aman 47% harusnya dropout juga tp gabisa ditentuin model

trs untuk matriks
14001 itu adalah aman yang beneran aman
2943 itu aman tp ditebak model jadi dropout
3781 itu dropout tp ditebak aman
4275 itu dropout yang beneran dropout

In [ ]:
deteksi_fitur= model.feature_importances_ 
df_fitur= pd.DataFrame({'Fitur': ['assignments_submitted_rate', 'units_failed', 'attendance_rate'], 'importance_score': deteksi_fitur})
df_fitur=df_fitur.round(2).sort_values(by='importance_score', ascending=False)
print(df_fitur)

feature_importances_ adalah fitur bawaan dari random forest, untuk ngecek manakah fitur yang paling berpengaruh terhadap target.
trs cara pakenya itu bikin variabel baru =pd.DataFrame({masukin fitur yg mau dicek, importance_skor=model.feature_importances_ ituu})
abistu urutin pake sort_values berdasarkan importance_skor, trs round itu buat 2 angka diblkg koma.

hasilnya yaitu yang paling berpengaruh thdp target dropout yaitu submit tugas dgn presentase 40%, kemudian kehadiran 39% dan unit failed 21%.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid={
    'n_estimators': [50,100,200],
    'max_depth': [10,20, None],
    'min_samples_split': [2,5,10]
}
# model = randomforestclassifier yg udh dibikin diatas
grid_search = GridSearchCV(estimator= model, param_grid =param_grid, cv=5, scoring='accuracy', n_jobs=1)
grid_search.fit(x_train,y_train)

print("parameter terbaik: ", grid_search.best_params_)
print('akurasi terbaik:', grid_search.best_score_)

cv=5, itu berati data dibagi menjadi 5 bagian, 4 untuk data latih dan 1 untuk data uji supaya bener" akurat.
n_jobs=1, itu berati kita pake 1 inti cpu untuk melatih dan menguji model, kl mau pake seluruh cpu itu n_jobs= -1
scoring='accuracy', dia emg bentuknya harus string, bisa di isi recall, precision atau f1

hasil nya didapatkan bahwa parameter terbaik yaitu dengan n_estimators 200, max_dept 10 dan min_samples_split 5
akurasi skor 76%, yang dimana naik dari 73% menjadi 76%

In [ ]:
model_terbaik = grid_search.best_estimator_
y_pred_baru = model_terbaik.predict(x_test)
print('classification report:\n', classification_report(y_test, y_pred_baru))
print('confusion matrix:\n', confusion_matrix(y_test, y_pred_baru))

bikin variabel model_terbaik buat cari best estimator nya, trs diuji lagi dengan model_terbaik.
hasil dari clasification report
kls 0 (aman). 80% dianggap beneran aman, 88% baru bisa ditebak aman 12% masi ditebak dropout
kls 1 (dropout). 69% beneran dropout, 53% baru bisa ditebak dropout krg 47% yg masi aman
hasil confusion matrix
14991 itu yg beneran aman
1953 harusnya aman tp dianggap dropout
3788 harusnya dropout tp dianggap aman
4268 beneran dropout